# Fire2Air Darwin — Assessment 3
## 04 Model 3 Count Prediction and Explainability — AWS Notebook
**Member:** Navodya Piumanthi

## Cell 1 — Imports and optional modelling packages
Loads the modelling, explainability and AWS dependencies. XGBoost and SHAP remain optional so the notebook can still run if either package is unavailable.

In [2]:
from pathlib import Path
import json
import joblib
import shutil
import tempfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import PoissonRegressor, TweedieRegressor
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_poisson_deviance,
)

import boto3

try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except Exception as exc:
    HAS_XGBOOST = False
    print("XGBoost unavailable:", exc)

try:
    import shap
    HAS_SHAP = True
except Exception as exc:
    HAS_SHAP = False
    print("SHAP unavailable:", exc)

print("XGBoost available:", HAS_XGBOOST)
print("SHAP available:", HAS_SHAP)


XGBoost available: True
SHAP available: True


In [18]:
%pip install "botocore[crt]"

  Using cached awscrt-0.36.0-cp313-abi3-win_amd64.whl.metadata (11 kB)
Using cached awscrt-0.36.0-cp313-abi3-win_amd64.whl (4.3 MB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
import awscrt
import botocore

print("AWS CRT installed successfully")
print("Botocore version:", botocore.__version__)

AWS CRT installed successfully
Botocore version: 1.43.93


## Cell 2 — Shared AWS S3 configuration and helper functions
Uses the same bucket/prefix structure as the other Fire2Air AWS scripts and adds reusable download, upload and listing helpers.

In [3]:
AWS_REGION = "ap-southeast-2"
BUCKET = "fire2air-darwin-dan5-646468992009-ap-southeast-2-an"


NTEPA_RAW = f"s3://{BUCKET}/raw/ntepa/"
FIRMS_RAW = f"s3://{BUCKET}/raw/firms/"
NTEPA_DAILY = f"s3://{BUCKET}/processed/ntepa_daily/"
FIRMS_DAILY = f"s3://{BUCKET}/processed/firms_daily/"
MODEL_READY = f"s3://{BUCKET}/processed/model_ready/"

MODEL1_DIR = f"s3://{BUCKET}/models/model1/"
MODEL2_DIR = f"s3://{BUCKET}/models/model2/"
MODEL3_DIR = f"s3://{BUCKET}/models/model3/"

MODEL1_OUTPUT = f"s3://{BUCKET}/outputs/model1/"
MODEL2_OUTPUT = f"s3://{BUCKET}/outputs/model2/"
MODEL3_OUTPUT = f"s3://{BUCKET}/outputs/model3/"

VIS_OUTPUT = f"s3://{BUCKET}/outputs/visualisation/"
DASHBOARD_DIR = f"s3://{BUCKET}/dashboard/"
DOCUMENTATION_DIR = f"s3://{BUCKET}/documentation/"
ATHENA_RESULTS = f"s3://{BUCKET}/athena-results/"

session = boto3.Session(
    profile_name="fire2air",
    region_name=AWS_REGION,
)

s3 = session.client("s3")

# Cross-platform temporary workspace: works on Windows, macOS and Linux.
LOCAL_ROOT = Path(tempfile.gettempdir()) / "fire2air_a3"
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

def _prefix(uri):
    return uri.replace(f"s3://{BUCKET}/", "", 1)

def download_prefix(uri, local_dir, flatten=False):
    local_dir = Path(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)
    prefix = _prefix(uri)
    paginator = s3.get_paginator("list_objects_v2")
    downloaded = []

    for page in paginator.paginate(Bucket=BUCKET, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.endswith("/"):
                continue

            rel = Path(key[len(prefix):].lstrip("/"))
            dest = local_dir / (rel.name if flatten else rel)
            dest.parent.mkdir(parents=True, exist_ok=True)

            s3.download_file(BUCKET, key, str(dest))
            downloaded.append(dest)

    return downloaded

def upload_file(local_path, uri, object_name=None):
    local_path = Path(local_path)
    prefix = _prefix(uri).rstrip("/")
    key = f"{prefix}/{object_name or local_path.name}"
    s3.upload_file(str(local_path), BUCKET, key)
    print(f"Uploaded: s3://{BUCKET}/{key}")
    return key

def upload_tree(local_dir, uri):
    local_dir = Path(local_dir)
    if not local_dir.exists():
        return []

    prefix = _prefix(uri).rstrip("/")
    uploaded = []

    for p in local_dir.rglob("*"):
        if p.is_file():
            rel = p.relative_to(local_dir).as_posix()
            key = f"{prefix}/{rel}"
            s3.upload_file(str(p), BUCKET, key)
            uploaded.append(key)

    print(f"Uploaded {len(uploaded)} file(s): {local_dir} -> {uri}")
    return uploaded

def list_prefix(uri):
    prefix = _prefix(uri)
    paginator = s3.get_paginator("list_objects_v2")
    rows = []

    for page in paginator.paginate(Bucket=BUCKET, Prefix=prefix):
        for obj in page.get("Contents", []):
            if obj["Key"].endswith("/"):
                continue
            rows.append({
                "Key": obj["Key"],
                "Size_Bytes": obj["Size"],
                "Last_Modified": obj["LastModified"],
            })

    return pd.DataFrame(rows)


## Cell 3 — AWS preflight, download shared model-ready data and initialise workspace
Checks S3 access, verifies the required shared checkpoint/manifest exist, downloads them, creates the local output folders, and loads the Model 3 target and feature manifest.

In [4]:
RANDOM_STATE = 42
BOOTSTRAP_REPEATS = 300

project_folder = LOCAL_ROOT / "model3_workspace"
if project_folder.exists():
    shutil.rmtree(project_folder)
project_folder.mkdir(parents=True, exist_ok=True)

# Confirm the project bucket is accessible.
s3.head_bucket(Bucket=BUCKET)

model_ready_objects = list_prefix(MODEL_READY)
if model_ready_objects.empty:
    raise FileNotFoundError(f"No objects found under {MODEL_READY}")

required_files = {
    "Fire2Air_model_ready_checkpoint_A3.csv",
    "Fire2Air_feature_manifest_A3.json",
}
available_names = {Path(key).name for key in model_ready_objects["Key"]}
missing_required = required_files.difference(available_names)

if missing_required:
    raise FileNotFoundError(
        f"Required Model 3 input file(s) missing from {MODEL_READY}: "
        f"{sorted(missing_required)}"
    )

print("Model-ready S3 objects:")
display(model_ready_objects)

downloaded_files = download_prefix(
    MODEL_READY,
    project_folder,
    flatten=True,
)
print(f"Downloaded {len(downloaded_files)} file(s) to:", project_folder)

output_root = project_folder / "outputs_prt661_a3"
table_dir = output_root / "tables"
figure_dir = output_root / "figures"
model_dir = output_root / "models"

for folder in [table_dir, figure_dir, model_dir]:
    folder.mkdir(parents=True, exist_ok=True)

data_path = project_folder / "Fire2Air_model_ready_checkpoint_A3.csv"
manifest_path = project_folder / "Fire2Air_feature_manifest_A3.json"

integrated_daily = pd.read_csv(
    data_path,
    parse_dates=["date", "target_date"],
)

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

model_features = manifest["features"]
feature_groups = manifest["feature_groups"]
duration_target = manifest["targets"]["count"]

print("AWS staging folder:", project_folder)
print("Dataset shape:", integrated_daily.shape)
print("Predictor count:", len(model_features))
print("Count target:", duration_target)


Model-ready S3 objects:


,Key,Size_Bytes,Last_Modified
0,processed/model_ready/Fire2Air_ablation_featur...,11203,2026-10-02 17:50:39+00:00
1,processed/model_ready/Fire2Air_feature_manifes...,15765,2026-10-02 17:50:39+00:00
2,processed/model_ready/Fire2Air_integrated_dail...,8164937,2026-10-02 17:50:35+00:00
3,processed/model_ready/Fire2Air_model_ready_che...,8116877,2026-10-02 17:50:37+00:00


Downloaded 4 file(s) to: C:\Users\navod\AppData\Local\Temp\fire2air_a3\model3_workspace
AWS staging folder: C:\Users\navod\AppData\Local\Temp\fire2air_a3\model3_workspace
Dataset shape: (7530, 120)
Predictor count: 93
Count target: target_hours_ge_25_next_day


## Cell 4 — Chronological split and target validation
Creates the leakage-safe training/validation/test periods and confirms the Model 3 target is physically valid from 0 to 24 hours.


In [5]:
duration_data = integrated_daily.dropna(
    subset=[duration_target, "target_date"]
).copy()

duration_data["target_year"] = duration_data["target_date"].dt.year

train_duration = duration_data[
    duration_data["target_year"] <= 2022
].copy()

validation_duration = duration_data[
    duration_data["target_year"] == 2023
].copy()

test_duration = duration_data[
    duration_data["target_year"] == 2024
].copy()

if min(
    len(train_duration),
    len(validation_duration),
    len(test_duration),
) == 0:
    raise ValueError(
        f"Insufficient chronological data: "
        f"train={len(train_duration)}, "
        f"validation={len(validation_duration)}, "
        f"test={len(test_duration)}"
    )

assert (
    train_duration["target_date"].max()
    < validation_duration["target_date"].min()
)

assert (
    validation_duration["target_date"].max()
    < test_duration["target_date"].min()
)

assert duration_data[duration_target].between(0, 24).all()

X_train = train_duration[model_features]
y_train = train_duration[duration_target]

X_validation = validation_duration[model_features]
y_validation = validation_duration[duration_target]

X_test = test_duration[model_features]
y_test = test_duration[duration_target]

print(
    "Split sizes:",
    {
        "train_2018_2022": len(train_duration),
        "validation_2023": len(validation_duration),
        "test_2024": len(test_duration),
    },
)
print(
    "Target range:",
    float(duration_data[duration_target].min()),
    "to",
    float(duration_data[duration_target].max()),
)


Split sizes: {'train_2018_2022': 2415, 'validation_2023': 661, 'test_2024': 645}
Target range: 0.0 to 24.0


## Cell 5 — Preprocessing, prediction bounds and count metrics

Defines reusable preprocessing and the Model 3 evaluation metrics, including MAE, RMSE, R² and mean Poisson deviance.

In [6]:
def onehot():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )


def build_preprocessor(features, scale_numeric=False):
    categorical = [f for f in features if f == "station"]
    numeric = [f for f in features if f not in categorical]

    transformers = []

    if categorical:
        transformers.append((
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent"),
                ),
                ("onehot", onehot()),
            ]),
            categorical,
        ))

    numeric_steps = [
        ("imputer", SimpleImputer(strategy="median"))
    ]

    if scale_numeric:
        numeric_steps.append(
            ("scaler", StandardScaler())
        )

    if numeric:
        transformers.append((
            "numeric",
            Pipeline(numeric_steps),
            numeric,
        ))

    return ColumnTransformer(transformers)


def bounded(predictions):
    return np.clip(
        np.asarray(predictions, dtype=float),
        0,
        24,
    )


def count_metrics(y_true, predictions):
    y = np.asarray(y_true, dtype=float)
    pred = bounded(predictions)

    return {
        "MAE": mean_absolute_error(y, pred),
        "RMSE": float(
            np.sqrt(mean_squared_error(y, pred))
        ),
        "R2": (
            r2_score(y, pred)
            if len(y) >= 2
            else np.nan
        ),
        "Mean_Poisson_Deviance": mean_poisson_deviance(
            y,
            np.clip(pred, 1e-6, None),
        ),
        "Prediction_Min": float(pred.min()),
        "Prediction_Max": float(pred.max()),
    }


def duration_strata_metrics(y_true, predictions):
    y = np.asarray(y_true, dtype=float)
    pred = bounded(predictions)

    strata = [
        ("Zero hours", y == 0),
        ("Low duration 1-5 h", (y >= 1) & (y <= 5)),
        ("Medium duration 6-12 h", (y >= 6) & (y <= 12)),
        ("Long duration 13-24 h", y >= 13),
    ]

    rows = []

    for label, mask in strata:
        if mask.sum() > 0:
            rows.append({
                "Stratum": label,
                "Rows": int(mask.sum()),
                "MAE": mean_absolute_error(
                    y[mask],
                    pred[mask],
                ),
                "RMSE": float(
                    np.sqrt(
                        mean_squared_error(
                            y[mask],
                            pred[mask],
                        )
                    )
                ),
                "Bias_PredMinusObs": float(
                    np.mean(pred[mask] - y[mask])
                ),
            })

    return rows


## Cell 6 — Exploratory two-stage Hurdle Random Forest

Implements the Assessment 3 two-stage benchmark: first estimate whether any elevated hour occurs, then estimate positive duration.
